# IntZ Example 15: Statistical Test of Sign Reversal ⭐ EPS Science
**EPS Research — Flynn & Cannaliato (2025)**

Formal statistical comparison between z=0 (SPARC, all positive) and z~0.9 (IntZ/KROSS, all negative) omega distributions.

Tests: Mann-Whitney U, effect size (Cohen's d), bootstrap confidence interval.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

# Load IntZ corpus
with open('/home/david/Documents/RAG Project/Z=2 RAG/intz_corpus_v1b.json') as f:
    data = json.load(f)
galaxies = data['galaxies']
print(f"Loaded {len(galaxies)} galaxies")

In [ ]:
from scipy.stats import mannwhitneyu
import statistics

# IntZ omega
omega_intz = [g['omega']['omega_value_rad_gyr'] for g in galaxies
              if g['omega']['omega_available']
              and g['omega']['omega_value_rad_gyr'] is not None]

# SPARC z=0 reference values (Flynn & Cannaliato 2025)
# Mean +7.06, SD 3.26, N=84 — simulate distribution
np.random.seed(42)
omega_sparc_sim = np.random.normal(7.06, 3.26, 84).tolist()

# Mann-Whitney U test
stat, p = mannwhitneyu(omega_sparc_sim, omega_intz, alternative='greater')
print(f"Mann-Whitney U = {stat:.1f}, p = {p:.2e}")
print(f"SPARC median:  +{statistics.median(omega_sparc_sim):.2f} rad/Gyr")
print(f"IntZ median:    {statistics.median(omega_intz):.2f} rad/Gyr")
print(f"Difference:     {statistics.median(omega_sparc_sim)-statistics.median(omega_intz):.2f} rad/Gyr")

# Cohen's d
def cohen_d(a, b):
    na, nb = len(a), len(b)
    pooled_sd = np.sqrt(((na-1)*np.std(a)**2 + (nb-1)*np.std(b)**2) / (na+nb-2))
    return (np.mean(a) - np.mean(b)) / pooled_sd

d = cohen_d(omega_sparc_sim, omega_intz)
print(f"Cohen's d = {d:.2f} (large effect: d>0.8)")

# Sign test
n_neg_intz = sum(1 for o in omega_intz if o < 0)
n_pos_sparc = sum(1 for o in omega_sparc_sim if o > 0)
print(f"\nSign test:")
print(f"  SPARC positive: {n_pos_sparc}/84 ({100*n_pos_sparc/84:.0f}%)")
print(f"  IntZ negative:  {n_neg_intz}/{len(omega_intz)} ({100*n_neg_intz/len(omega_intz):.0f}%)")

In [ ]:
# Visualise the two distributions together
fig, ax = plt.subplots(figsize=(9,4))
ax.hist(omega_sparc_sim, bins=20, alpha=0.7, color='green',
        label=f'SPARC z=0 (simulated, N=84, mean=+7.06)')
ax.hist(omega_intz, bins=25, alpha=0.7, color='firebrick',
        label=f'IntZ z~0.9 (N={len(omega_intz)}, median={statistics.median(omega_intz):.2f})')
ax.axvline(0, color='black', ls='-', lw=1.5, alpha=0.7, label='ω = 0')
ax.set_xlabel('ω (rad/Gyr)')
ax.set_ylabel('N galaxies')
ax.set_title('Sign Reversal: z=0 vs z~0.9 Omega Distributions')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('intz_nb15_sign_reversal.png', dpi=120)
plt.show()